In [ ]:
import json
import csv
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

results_file = Path("./outputs/results.csv")

tasks_setup = {
    "rte":  {"lr": 1e-3, "metric": "Accuracy"},
    "mrpc": {"lr": 7e-4, "metric": "F1"},
    "cola": {"lr": 7e-4, "metric": "MCC"},
    "qnli": {"lr": 1e-4, "metric": "Accuracy"}
}
#  bert fullft: rte 2e-5
#  bert fullt: cola 2e-5
#  bert bitfit: cola 7e-4

#  dbert fullft: qnli 2e-4
#  dbert bitfit: qnli 5e-4
#  dbert bitfit: rte 3e-5
#  dbert fullft: rte 3e-5
#  dbert fullft: mrpc 1e-5

In [ ]:
!pip install torch transformers datasets tokenizers scikit-learn scipy sentencepiece sacremoses tqdm matplotlib seaborn pandas numpy

In [2]:
def run_task(task, 
             fine_tune_type="bitfit", 
             model="bert-base-cased", 
             seeds=[1, 2, 3, 4, 5], 
             batch=16, 
             gpu=0,
             lr="2e-5",
             epoch="20"):
             
    # On expérimente / reproduit que sur BERT Base et DBert ici
    model_name = "bb" if model=="bert-base-cased" else "d-bert"

    config = tasks_setup[task]
    metric = config["metric"]
    results = []

    for seed in seeds:
        output_path = Path("./outputs") / f"{model_name}_{fine_tune_type}_{task}_seed{seed}"
        output_path.mkdir(parents=True, exist_ok=True)

        subprocess.run(["python", "run_glue.py",
                        "--output-path", str(output_path),
                        "--task-name", task,
                        "--model-name", model,
                        "--fine-tune-type", fine_tune_type,
                        "--learning-rate", str(lr),
                        "--epochs", str(epoch),
                        "--batch-size", str(batch),
                        "--gpu-device", str(gpu),
                        "--seed", str(seed),
                        "--verbose"])

        with open(output_path / "eval_results.json") as f:
            acc = json.load(f)["validation"][metric]
        results.append(acc)

    mean = np.mean(results) * 100
    std = np.std(results) * 100

    results_file.parent.mkdir(parents=True, exist_ok=True)
    write_header = not results_file.exists()
    with open(results_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["model", "task", "fine_tune_type", "metric", "mean", "std"])
        if write_header:
            writer.writeheader()
        writer.writerow({"model": model, 
                         "task": task, 
                         "fine_tune_type": fine_tune_type,
                         "metric": metric, 
                         "mean": mean, 
                         "std": std})

In [8]:
run_task("rte", fine_tune_type="bitfit", model="distilbert-base-cased", lr=1e-5, epoch=20)

2026-03-19 19:46:35,316 - run_glue - INFO - ############################################################################################
2026-03-19 19:46:35,316 - run_glue - INFO - ############################################################################################
2026-03-19 19:46:35,316 - run_glue - INFO - ############################################################################################
2026-03-19 19:46:35,316 - run_glue - INFO - 
2026-03-19 19:46:35,316 - run_glue - INFO - Training Details: 
2026-03-19 19:46:35,316 - run_glue - INFO - ----------------------------------------------
2026-03-19 19:46:35,316 - run_glue - INFO - Model Name: distilbert-base-cased
2026-03-19 19:46:35,316 - run_glue - INFO - Task Name: rte
2026-03-19 19:46:35,316 - run_glue - INFO - Fine Tuning Type: bitfit
2026-03-19 19:46:35,316 - run_glue - INFO - Output Directory: outputs/d-bert_bitfit_rte_seed1
2026-03-19 19:46:35,316 - run_glue - INFO - Running on GPU #0
2026-03-19 19:46:35,316 - ru

2026-03-19 19:46:36,697 - _client - INFO - HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=glue "HTTP/1.1 404 Not Found"
2026-03-19 19:46:36,799 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/ax?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-19 19:46:36,902 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/ax?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-19 19:46:37,001 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-19 19:46:37,099 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c?recursive=false&expand=false "HTTP/1.1 200 OK"
2026-03-19 19:46:37,210 -

Map: 4980 examples [00:00, 5386.91 examples/s]         
Map: 554 examples [00:00, 4110.65 examples/s]        
Map: 6000 examples [00:00, 5567.42 examples/s]         


2026-03-19 19:46:43,453 - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5901.57it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map: 4980 examples [00:00, 4544.45 examples/s]         
Map: 554 examples [00:00, 4016.31 examples/s]        
Map: 6000 examples [00:00, 5124.94 examples/s]         


2026-03-19 20:29:52,968 - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3224.40it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.




Trainable Components:
----------------------------------------

distilbert.embeddings.LayerNorm.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.q_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.k_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.v_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.attention.out_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.sa_layer_norm.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.ffn.lin1.bias   --->   torch.Size([3072])
distilbert.transformer.layer.0.ffn.lin2.bias   --->   torch.Size([768])
distilbert.transformer.layer.0.output_layer_norm.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.q_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.k_lin.bias   --->   torch.Size([768])
distilbert.transformer.layer.1.attention.v_lin.bias   --->   torch.Size([768])
distilbert.t

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6521.10it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
